<a href="https://colab.research.google.com/github/Vyshnavi0918/Vyshnavi/blob/main/IDS_Final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from groq import Groq

# --------------------------------------------------
# PAGE CONFIG
# --------------------------------------------------
st.set_page_config(
    page_title="AI-Based NIDS | ML Comparison",
    layout="wide"
)

st.title("AI-Based Network Intrusion Detection System")
st.caption("Streamlit App with ML Model Comparison, Attack Detection & Explainable AI")

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
DATA_FILE = "/content/UNSW_NB15_training-set.csv"

FEATURES = [
    'dur',        # Duration
    'spkts',      # Source packets
    'dpkts',      # Destination packets
    'sbytes',     # Source bytes
    'dbytes',     # Destination bytes
    'sttl',       # Source TTL
    'dttl',       # Destination TTL
    'sload',      # Source load
    'dload'       # Destination load
]

# --------------------------------------------------
# SIDEBAR
# --------------------------------------------------
st.sidebar.header("🔑 Settings")
groq_api_key = st.sidebar.text_input("Groq API Key", type="password")

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
@st.cache_data
def load_data():
    df = pd.read_csv(DATA_FILE, encoding='ISO-8859-1', low_memory=False, nrows=20000)
    df.columns = df.columns.str.strip().str.lower()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    df['attack_cat'] = df['attack_cat'].astype(str)
    return df

df = load_data()
st.sidebar.success(f"Dataset Loaded: {len(df)} rows")

# --------------------------------------------------
# DISPLAY MODELS BEING COMPARED
# --------------------------------------------------
st.sidebar.markdown("### Models Compared")
st.sidebar.markdown(
    "- Logistic Regression\n" +
    "- Decision Tree\n" +
    "- Random Forest\n" +
    "- SVM (Support Vector Machine)\n" +
    "- KNN (K-Nearest Neighbors)"
)

# --------------------------------------------------
# TRAIN BUTTON
# --------------------------------------------------
if st.sidebar.button("🚀 Train & Compare Models"):

    with st.spinner("Training models..."):

        df_filtered = df[df['attack_cat'] != '0'].copy()
        X = df_filtered[FEATURES]
        y = df_filtered["attack_cat"]

        value_counts = y.value_counts()
        single_sample_classes = value_counts[value_counts < 2].index.tolist()
        if single_sample_classes:
            y_filtered_for_split = y[~y.isin(single_sample_classes)]
            X_filtered_for_split = X.loc[y_filtered_for_split.index]
        else:
            y_filtered_for_split = y
            X_filtered_for_split = X

        if len(y_filtered_for_split.unique()) > 1:
            X_train, X_test, y_train, y_test = train_test_split(
                X_filtered_for_split,
                y_filtered_for_split,
                test_size=0.3,
                random_state=42,
                stratify=y_filtered_for_split
            )
        else:
            st.error("Not enough data or classes left to perform a meaningful split.")
            st.stop()

        models = {
            "Logistic Regression": Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1000))
            ]),
            "Decision Tree": DecisionTreeClassifier(max_depth=10),
            "Random Forest": RandomForestClassifier(
                n_estimators=100, max_depth=12, random_state=42
            ),
            "SVM": Pipeline([
                ("scaler", StandardScaler()),
                ("model", SVC(probability=True))
            ]),
            "KNN": Pipeline([
                ("scaler", StandardScaler()),
                ("model", KNeighborsClassifier(n_neighbors=5))
            ])
        }

        results = []

        for name, model in models.items():
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

            report = classification_report(
                y_test, preds, output_dict=True, zero_division=0
            )

            results.append({
                "Model": name,
                "Accuracy": accuracy_score(y_test, preds),
                "Precision": report["weighted avg"]["precision"],
                "Recall": report["weighted avg"]["recall"],
                "F1-Score": report["weighted avg"]["f1-score"]
            })

        results_df = pd.DataFrame(results).sort_values(
            "Accuracy", ascending=False
        )

        best_model_name = results_df.iloc[0]["Model"]
        best_model = models[best_model_name]
        best_model.fit(X_train, y_train) # Re-fit best model on full train data

        st.session_state.update({
            "results_df": results_df,
            "best_model": best_model,
            "best_model_name": best_model_name,
            "X_test": X_test,
            "y_test": y_test
        })

# --------------------------------------------------
# DISPLAY RESULTS
# --------------------------------------------------
if "results_df" in st.session_state:

    # ---------------- MODEL COMPARISON TABLE ----------------
    st.header("1️⃣ Machine Learning Model Comparison")
    st.dataframe(
        st.session_state["results_df"],
        use_container_width=True
    )

    st.success(
        f"🏆 Best Model Selected: "
        f"**{st.session_state['best_model_name']}**"
    )

    # ---------------- BAR CHART ----------------
    st.subheader("📊 Model Performance Comparison")

    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    melted = st.session_state["results_df"].melt(
        id_vars="Model",
        value_vars=metrics,
        var_name="Metric",
        value_name="Score"
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=melted,
        x="Metric",
        y="Score",
        hue="Model",
        ax=ax
    )
    ax.set_ylim(0, 1)
    ax.set_title("ML Model Performance Comparison")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    st.pyplot(fig)

    # ---------------- CONFUSION MATRIX ----------------
    st.subheader("Confusion Matrix (Best Model)")
    y_pred = st.session_state["best_model"].predict(
        st.session_state["X_test"]
    )

    cm = confusion_matrix(
        st.session_state["y_test"], y_pred
    )

    fig, ax = plt.subplots(figsize=(8, 6)) # Increased figsize for better design
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=np.unique(st.session_state["y_test"]),
        yticklabels=np.unique(st.session_state["y_test"]),
        cbar=True, # Explicitly show color bar
        annot_kws={"size": 10} # Adjust annotation font size
    )
    ax.set_xlabel("Predicted Label", fontsize=12) # Added fontsize
    ax.set_ylabel("True Label", fontsize=12) # Added fontsize, changed to True Label for clarity
    ax.set_title("Confusion Matrix for Best Model", fontsize=14) # Added explicit title and fontsize
    st.pyplot(fig)

    # ---------------- ATTACK VS BENIGN ----------------
    st.header("2️⃣ Attack vs Benign Examples")

    benign = df[df["attack_cat"] == "Normal"].sample(1)
    attack = df[df["attack_cat"] != "Normal"].sample(1)

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("🟢 BENIGN Packet")
        st.dataframe(benign[FEATURES].T)
        st.success("Normal traffic behavior")

    with col2:
        st.subheader("🔴 ATTACK Packet")
        st.dataframe(attack[FEATURES].T)
        st.error("Malicious traffic behavior")

    # ---------------- LIVE SIMULATION ----------------
    st.header("3️⃣ Live Packet Detection")

    if st.button("🎲 Capture Random Packet"):
        idx = np.random.randint(0, len(st.session_state["X_test"]))
        st.session_state["packet"] = st.session_state["X_test"].iloc[idx]
        st.session_state["actual"] = st.session_state["y_test"].iloc[idx]

    if "packet" in st.session_state:
        packet = st.session_state["packet"]

        col1, col2 = st.columns(2)

        with col1:
            st.subheader("Packet Features")
            st.dataframe(packet.to_frame(), use_container_width=True) # Display features vertically

        with col2:
            pred = st.session_state["best_model"].predict([packet])[0]
            st.subheader("Detection Result")

            # Create a small DataFrame for the result summary as requested
            result_summary_df = pd.DataFrame({
                "Category": ["Predicted Label", "Actual Label"],
                "Value": [pred, st.session_state['actual']]
            })
            st.dataframe(result_summary_df, hide_index=True, use_container_width=True)

            if pred == "Normal":
                st.success("SAFE (NORMAL)")
            else:
                st.error(f"🚨 ATTACK DETECTED ({pred})")

            # ---------------- GROQ AI ----------------
            st.markdown("---")
            st.subheader("Ask AI Analyst")

            if st.button("Explain with AI"):
                if not groq_api_key:
                    st.warning("Please enter Groq API key")
                else:
                    client = Groq(api_key=groq_api_key)

                    prompt = f"""
                    You are a cybersecurity analyst.

                    Prediction: {pred}
                    Actual Label: {st.session_state['actual']}

                    Packet values:
                    {packet.to_string()}

                    Explain why this traffic is benign or an attack.
                    Suggest remediation and preventive measures.
                    Keep it simple for a student.
                    """

                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=[{"role": "user", "content": prompt}],
                        temperature=0.5
                    )

                    st.info(response.choices[0].message.content)

    # ---------------- SHAP EXPLAINABILITY ----------------
    st.header("4️⃣ Explainable AI (SHAP Feature Impact)")

    best_model = st.session_state["best_model"]
    best_model_name = st.session_state["best_model_name"]

    # SHAP only for tree-based models
    if best_model_name in ["Random Forest", "Decision Tree"]:

        with st.spinner("Generating SHAP explanation..."):

            # Handle Pipeline vs direct model
            if hasattr(best_model, "named_steps"):
                tree_model = best_model.named_steps.get("model", None)
            else:
                tree_model = best_model

            # Ensure robust sampling of X_test for SHAP explanation
            num_shap_samples = min(200, len(st.session_state["X_test"])) # Using 200 samples for reasonable performance
            if num_shap_samples > 0:
                X_sample = st.session_state["X_test"].sample(n=num_shap_samples, random_state=42)

                # If the model is a pipeline and has a scaler, transform X_sample for SHAP
                X_sample_display = X_sample
                if hasattr(best_model, 'named_steps') and 'scaler' in best_model.named_steps:
                    scaler = best_model.named_steps['scaler']
                    X_sample_processed = scaler.transform(X_sample)
                    X_sample_display = pd.DataFrame(X_sample_processed, columns=X_sample.columns, index=X_sample.index)

                # Create SHAP explainer
                explainer = shap.TreeExplainer(
                    tree_model,
                    feature_perturbation="tree_path_dependent" # Recommended for tree models
                )

                # Compute SHAP values
                shap_values = explainer.shap_values(X_sample_display)

                # ----- MULTI-CLASS HANDLING -----
                class_names = tree_model.classes_
                class_index = 0 # Default to the first class
                selected_class_name = class_names[0]

                # Try to find a 'Normal' or 'BENIGN' class to display first
                if "Normal" in class_names:
                    class_index = list(class_names).index("Normal")
                    selected_class_name = "Normal"
                elif "BENIGN" in class_names:
                    class_index = list(class_names).index("BENIGN")
                    selected_class_name = "BENIGN"

                shap_values_to_plot = None
                if isinstance(shap_values, list):
                    # For multi-output models, shap_values is a list of arrays
                    if len(shap_values) > class_index:
                        shap_values_to_plot = shap_values[class_index]
                elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
                    # For multi-class (softmax) models, shap_values can be a 3D array
                    if shap_values.shape[2] > class_index:
                        shap_values_to_plot = shap_values[:, :, class_index]
                else:
                    # For binary or single-output models, it's a 2D array
                    shap_values_to_plot = shap_values

                # ----- FORCE MATPLOTLIB PLOT -----
                if shap_values_to_plot is not None and len(shap_values_to_plot) > 0:
                    fig, ax = plt.subplots(figsize=(10, 6)) # Slightly increased height for labels
                    shap.summary_plot(
                        shap_values_to_plot,
                        X_sample_display,
                        plot_type="bar",
                        show=False # Prevents SHAP from calling plt.show() itself
                    )
                    ax.set_title(f"SHAP Feature Impact for '{best_model_name}' (Class: {selected_class_name})", fontsize=14)
                    ax.set_xlabel("SHAP Value (Impact on Model Output)", fontsize=12) # Added xlabel
                    ax.set_ylabel("Feature", fontsize=12) # Added ylabel
                    plt.tight_layout()
                    st.pyplot(fig)
                else:
                    st.info("No SHAP values to plot for the selected class or plot is empty.")

            else:
                st.info("Not enough samples in X_test to generate SHAP explanation.")

    else:
        st.info(
            "SHAP explanation is supported only for tree-based models." # Removed specific names for better generality
        )

else:
    st.info("👈 Click **Train & Compare Models** from the sidebar")

2026-02-08 18:05:36.699 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:05:36.701 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:05:36.908 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-02-08 18:05:36.909 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:05:36.911 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:05:36.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:05:36.915 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [2]:
%%writefile app.py
# The content of this cell will be written to a file named 'app.py'
# To ensure the Streamlit application runs correctly, the entire Streamlit script from the previous cell is included here.

import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from groq import Groq

# --------------------------------------------------
# PAGE CONFIG
# --------------------------------------------------
st.set_page_config(
    page_title="AI-Based NIDS | ML Comparison",
    layout="wide"
)

st.title("AI-Based Network Intrusion Detection System")
st.caption("Streamlit App with ML Model Comparison, Attack Detection & Explainable AI")

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
DATA_FILE = "/content/UNSW_NB15_training-set.csv"

FEATURES = [
    'dur',        # Duration
    'spkts',      # Source packets
    'dpkts',      # Destination packets
    'sbytes',     # Source bytes
    'dbytes',     # Destination bytes
    'sttl',       # Source TTL
    'dttl',       # Destination TTL
    'sload',      # Source load
    'dload'       # Destination load
]

# --------------------------------------------------
# SIDEBAR
# --------------------------------------------------
st.sidebar.header("🔑 Settings")
groq_api_key = st.sidebar.text_input("Groq API Key", type="password")

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
@st.cache_data
def load_data():
    df = pd.read_csv(DATA_FILE, encoding='ISO-8859-1', low_memory=False, nrows=20000)
    df.columns = df.columns.str.strip().str.lower()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    df['attack_cat'] = df['attack_cat'].astype(str)
    return df

df = load_data()
st.sidebar.success(f"Dataset Loaded: {len(df)} rows")

# --------------------------------------------------
# DISPLAY MODELS BEING COMPARED
# --------------------------------------------------
st.sidebar.markdown("### Models Compared")
st.sidebar.markdown(
    "- Logistic Regression\n" +
    "- Decision Tree\n" +
    "- Random Forest\n" +
    "- SVM (Support Vector Machine)\n" +
    "- KNN (K-Nearest Neighbors)"
)

# --------------------------------------------------
# TRAIN BUTTON
# --------------------------------------------------
if st.sidebar.button("🚀 Train & Compare Models"):

    with st.spinner("Training models..."):

        df_filtered = df[df['attack_cat'] != '0'].copy()
        X = df_filtered[FEATURES]
        y = df_filtered["attack_cat"]

        value_counts = y.value_counts()
        single_sample_classes = value_counts[value_counts < 2].index.tolist()
        if single_sample_classes:
            y_filtered_for_split = y[~y.isin(single_sample_classes)]
            X_filtered_for_split = X.loc[y_filtered_for_split.index]
        else:
            y_filtered_for_split = y
            X_filtered_for_split = X

        if len(y_filtered_for_split.unique()) > 1:
            X_train, X_test, y_train, y_test = train_test_split(
                X_filtered_for_split,
                y_filtered_for_split,
                test_size=0.3,
                random_state=42,
                stratify=y_filtered_for_split
            )
        else:
            st.error("Not enough data or classes left to perform a meaningful split.")
            st.stop()

        models = {
            "Logistic Regression": Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1000))
            ]),
            "Decision Tree": DecisionTreeClassifier(max_depth=10),
            "Random Forest": RandomForestClassifier(
                n_estimators=100, max_depth=12, random_state=42
            ),
            "SVM": Pipeline([
                ("scaler", StandardScaler()),
                ("model", SVC(probability=True))
            ]),
            "KNN": Pipeline([
                ("scaler", StandardScaler()),
                ("model", KNeighborsClassifier(n_neighbors=5))
            ])
        }

        results = []

        for name, model in models.items():
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

            report = classification_report(
                y_test, preds, output_dict=True, zero_division=0
            )

            results.append({
                "Model": name,
                "Accuracy": accuracy_score(y_test, preds),
                "Precision": report["weighted avg"]["precision"],
                "Recall": report["weighted avg"]["recall"],
                "F1-Score": report["weighted avg"]["f1-score"]
            })

        results_df = pd.DataFrame(results).sort_values(
            "Accuracy", ascending=False
        )

        best_model_name = results_df.iloc[0]["Model"]
        best_model = models[best_model_name]
        best_model.fit(X_train, y_train) # Re-fit best model on full train data

        st.session_state.update({
            "results_df": results_df,
            "best_model": best_model,
            "best_model_name": best_model_name,
            "X_test": X_test,
            "y_test": y_test
        })

# --------------------------------------------------
# DISPLAY RESULTS
# --------------------------------------------------
if "results_df" in st.session_state:

    # ---------------- MODEL COMPARISON TABLE ----------------
    st.header("1️⃣ Machine Learning Model Comparison")
    st.dataframe(
        st.session_state["results_df"],
        use_container_width=True
    )

    st.success(
        f"🏆 Best Model Selected: "
        f"**{st.session_state['best_model_name']}**"
    )

    # ---------------- BAR CHART ----------------
    st.subheader("📊 Model Performance Comparison")

    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    melted = st.session_state["results_df"].melt(
        id_vars="Model",
        value_vars=metrics,
        var_name="Metric",
        value_name="Score"
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=melted,
        x="Metric",
        y="Score",
        hue="Model",
        ax=ax
    )
    ax.set_ylim(0, 1)
    ax.set_title("ML Model Performance Comparison")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    st.pyplot(fig)

    # ---------------- CONFUSION MATRIX ----------------
    st.subheader("Confusion Matrix (Best Model)")
    y_pred = st.session_state["best_model"].predict(
        st.session_state["X_test"]
    )

    cm = confusion_matrix(
        st.session_state["y_test"], y_pred
    )

    fig, ax = plt.subplots(figsize=(8, 6)) # Increased figsize for better design
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=np.unique(st.session_state["y_test"]),
        yticklabels=np.unique(st.session_state["y_test"]),
        cbar=True, # Explicitly show color bar
        annot_kws={"size": 10} # Adjust annotation font size
    )
    ax.set_xlabel("Predicted Label", fontsize=12) # Added fontsize
    ax.set_ylabel("True Label", fontsize=12) # Added fontsize, changed to True Label for clarity
    ax.set_title("Confusion Matrix for Best Model", fontsize=14) # Added explicit title and fontsize
    st.pyplot(fig)

    # ---------------- ATTACK VS BENIGN ----------------
    st.header("2️⃣ Attack vs Benign Examples")

    benign = df[df["attack_cat"] == "Normal"].sample(1)
    attack = df[df["attack_cat"] != "Normal"].sample(1)

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("🟢 BENIGN Packet")
        st.dataframe(benign[FEATURES].T)
        st.success("Normal traffic behavior")

    with col2:
        st.subheader("🔴 ATTACK Packet")
        st.dataframe(attack[FEATURES].T)
        st.error("Malicious traffic behavior")

    # ---------------- LIVE SIMULATION ----------------
    st.header("3️⃣ Live Packet Detection")

    if st.button("🎲 Capture Random Packet"):
        idx = np.random.randint(0, len(st.session_state["X_test"]))
        st.session_state["packet"] = st.session_state["X_test"].iloc[idx]
        st.session_state["actual"] = st.session_state["y_test"].iloc[idx]

    if "packet" in st.session_state:
        packet = st.session_state["packet"]

        col1, col2 = st.columns(2)

        with col1:
            st.subheader("Packet Features")
            st.dataframe(packet.to_frame(), use_container_width=True) # Display features vertically

        with col2:
            pred = st.session_state["best_model"].predict([packet])[0]
            st.subheader("Detection Result")

            # Create a small DataFrame for the result summary as requested
            result_summary_df = pd.DataFrame({
                "Category": ["Predicted Label", "Actual Label"],
                "Value": [pred, st.session_state['actual']]
            })
            st.dataframe(result_summary_df, hide_index=True, use_container_width=True)

            if pred == "Normal":
                st.success("SAFE (NORMAL)")
            else:
                st.error(f"🚨 ATTACK DETECTED ({pred})")

            # ---------------- GROQ AI ----------------
            st.markdown("---")
            st.subheader("Ask AI Analyst")

            if st.button("Explain with AI"):
                if not groq_api_key:
                    st.warning("Please enter Groq API key")
                else:
                    client = Groq(api_key=groq_api_key)

                    prompt = f"""
                    You are a cybersecurity analyst.

                    Prediction: {pred}
                    Actual Label: {st.session_state['actual']}

                    Packet values:
                    {packet.to_string()}

                    Explain why this traffic is benign or an attack.
                    Suggest remediation and preventive measures.
                    Keep it simple for a student.
                    """

                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=[{"role": "user", "content": prompt}],
                        temperature=0.5
                    )

                    st.info(response.choices[0].message.content)

    # ---------------- SHAP EXPLAINABILITY ----------------
    st.header("4️⃣ Explainable AI (SHAP Feature Impact)")

    best_model = st.session_state["best_model"]
    best_model_name = st.session_state["best_model_name"]

    # SHAP only for tree-based models
    if best_model_name in ["Random Forest", "Decision Tree"]:

        with st.spinner("Generating SHAP explanation..."):

            # Handle Pipeline vs direct model
            if hasattr(best_model, "named_steps"):
                tree_model = best_model.named_steps.get("model", None)
            else:
                tree_model = best_model

            # Ensure robust sampling of X_test for SHAP explanation
            num_shap_samples = min(200, len(st.session_state["X_test"])) # Using 200 samples for reasonable performance
            if num_shap_samples > 0:
                X_sample = st.session_state["X_test"].sample(n=num_shap_samples, random_state=42)

                # If the model is a pipeline and has a scaler, transform X_sample for SHAP
                X_sample_display = X_sample
                if hasattr(best_model, 'named_steps') and 'scaler' in best_model.named_steps:
                    scaler = best_model.named_steps['scaler']
                    X_sample_processed = scaler.transform(X_sample)
                    X_sample_display = pd.DataFrame(X_sample_processed, columns=X_sample.columns, index=X_sample.index)

                # Create SHAP explainer
                explainer = shap.TreeExplainer(
                    tree_model,
                    feature_perturbation="tree_path_dependent" # Recommended for tree models
                )

                # Compute SHAP values
                shap_values = explainer.shap_values(X_sample_display)

                # ----- MULTI-CLASS HANDLING -----
                class_names = tree_model.classes_
                class_index = 0 # Default to the first class
                selected_class_name = class_names[0]

                # Try to find a 'Normal' or 'BENIGN' class to display first
                if "Normal" in class_names:
                    class_index = list(class_names).index("Normal")
                    selected_class_name = "Normal"
                elif "BENIGN" in class_names:
                    class_index = list(class_names).index("BENIGN")
                    selected_class_name = "BENIGN"

                shap_values_to_plot = None
                if isinstance(shap_values, list):
                    # For multi-output models, shap_values is a list of arrays
                    if len(shap_values) > class_index:
                        shap_values_to_plot = shap_values[class_index]
                elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
                    # For multi-class (softmax) models, shap_values can be a 3D array
                    if shap_values.shape[2] > class_index:
                        shap_values_to_plot = shap_values[:, :, class_index]
                else:
                    # For binary or single-output models, it's a 2D array
                    shap_values_to_plot = shap_values

                # ----- FORCE MATPLOTLIB PLOT -----
                if shap_values_to_plot is not None and len(shap_values_to_plot) > 0:
                    fig, ax = plt.subplots(figsize=(10, 6)) # Slightly increased height for labels
                    shap.summary_plot(
                        shap_values_to_plot,
                        X_sample_display,
                        plot_type="bar",
                        show=False # Prevents SHAP from calling plt.show() itself
                    )
                    ax.set_title(f"SHAP Feature Impact for '{best_model_name}' (Class: {selected_class_name})", fontsize=14)
                    ax.set_xlabel("SHAP Value (Impact on Model Output)", fontsize=12) # Added xlabel
                    ax.set_ylabel("Feature", fontsize=12) # Added ylabel
                    plt.tight_layout()
                    st.pyplot(fig)
                else:
                    st.info("No SHAP values to plot for the selected class or plot is empty.")

            else:
                st.info("Not enough samples in X_test to generate SHAP explanation.")

    else:
        st.info(
            "SHAP explanation is supported only for tree-based models." # Removed specific names for better generality
        )

else:
    st.info("👈 Click **Train & Compare Models** from the sidebar")

Overwriting app.py


In [3]:
!streamlit run app.py &>/dev/null&

In [4]:
# Install missing libraries
!pip install streamlit groq shap

In [5]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from groq import Groq

# --------------------------------------------------
# PAGE CONFIG
# --------------------------------------------------
st.set_page_config(
    page_title="AI-Based NIDS | ML Comparison",
    layout="wide"
)

st.title("AI-Based Network Intrusion Detection System")
st.caption("Streamlit App with ML Model Comparison, Attack Detection & Explainable AI")

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
DATA_FILE = "/content/UNSW_NB15_training-set.csv"

FEATURES = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Total Length of Fwd Packets',
    'Fwd Packet Length Max',
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow Packets/s'
]

# --------------------------------------------------
# SIDEBAR
# --------------------------------------------------
st.sidebar.header("🔑 Settings")
groq_api_key = st.sidebar.text_input("Groq API Key", type="password")

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
@st.cache_data
def load_data():
    df = pd.read_csv(DATA_FILE, nrows=20000)
    df.columns = df.columns.str.strip()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    return df

df = load_data()
st.sidebar.success(f"Dataset Loaded: {len(df)} rows")

# --------------------------------------------------
# DISPLAY MODELS BEING COMPARED
# --------------------------------------------------
st.sidebar.markdown("### Models Compared")
st.sidebar.markdown(
    "- Logistic Regression\n" +
    "- Decision Tree\n" +
    "- Random Forest\n" +
    "- SVM (Support Vector Machine)\n" +
    "- KNN (K-Nearest Neighbors)"
)

# --------------------------------------------------
# TRAIN BUTTON
# --------------------------------------------------
if st.sidebar.button("🚀 Train & Compare Models"):

    with st.spinner("Training models..."):

        X = df[FEATURES]
        y = df["Label"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.3,
            random_state=42,
            stratify=y
        )

        models = {
            "Logistic Regression": Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1000))
            ]),
            "Decision Tree": DecisionTreeClassifier(max_depth=10),
            "Random Forest": RandomForestClassifier(
                n_estimators=100, max_depth=12, random_state=42
            ),
            "SVM": Pipeline([
                ("scaler", StandardScaler()),
                ("model", SVC(probability=True))
            ]),
            "KNN": Pipeline([
                ("scaler", StandardScaler()),
                ("model", KNeighborsClassifier(n_neighbors=5))
            ])
        }

        results = []

        for name, model in models.items():
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

            report = classification_report(
                y_test, preds, output_dict=True
            )

            results.append({
                "Model": name,
                "Accuracy": accuracy_score(y_test, preds),
                "Precision": report["weighted avg"]["precision"],
                "Recall": report["weighted avg"]["recall"],
                "F1-Score": report["weighted avg"]["f1-score"]
            })

        results_df = pd.DataFrame(results).sort_values(
            "Accuracy", ascending=False
        )

        best_model_name = results_df.iloc[0]["Model"]
        best_model = models[best_model_name]
        best_model.fit(X_train, y_train)

        st.session_state.update({
            "results_df": results_df,
            "best_model": best_model,
            "best_model_name": best_model_name,
            "X_test": X_test,
            "y_test": y_test
        })

# --------------------------------------------------
# DISPLAY RESULTS
# --------------------------------------------------
if "results_df" in st.session_state:

    # ---------------- MODEL COMPARISON TABLE ----------------
    st.header("1️⃣ Machine Learning Model Comparison")
    st.dataframe(
        st.session_state["results_df"],
        use_container_width=True
    )

    st.success(
        f"🏆 Best Model Selected: "
        f"**{st.session_state['best_model_name']}**"
    )

    # ---------------- BAR CHART ----------------
    st.subheader("📊 Model Performance Comparison")

    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    melted = st.session_state["results_df"].melt(
        id_vars="Model",
        value_vars=metrics,
        var_name="Metric",
        value_name="Score"
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=melted,
        x="Metric",
        y="Score",
        hue="Model",
        ax=ax
    )
    ax.set_ylim(0, 1)
    ax.set_title("ML Model Performance Comparison")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    st.pyplot(fig)

    # ---------------- CONFUSION MATRIX ----------------
    st.subheader("Confusion Matrix (Best Model)")
    y_pred = st.session_state["best_model"].predict(
        st.session_state["X_test"]
    )

    cm = confusion_matrix(
        st.session_state["y_test"], y_pred
    )

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=np.unique(st.session_state["y_test"]),
        yticklabels=np.unique(st.session_state["y_test"])
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    st.pyplot(fig)

    # ---------------- ATTACK VS BENIGN ----------------
    st.header("2️⃣ Attack vs Benign Examples")

    benign = df[df["Label"] == "BENIGN"].sample(1)
    attack = df[df["Label"] != "BENIGN"].sample(1)

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("🟢 BENIGN Packet")
        st.dataframe(benign[FEATURES].T)
        st.success("Normal traffic behavior")

    with col2:
        st.subheader("🔴 ATTACK Packet")
        st.dataframe(attack[FEATURES].T)
        st.error("Malicious traffic behavior")

    # ---------------- LIVE SIMULATION ----------------
    st.header("3️⃣ Live Packet Detection")

    if st.button("🎲 Capture Random Packet"):
        idx = np.random.randint(0, len(st.session_state["X_test"]))
        st.session_state["packet"] = st.session_state["X_test"].iloc[idx]
        st.session_state["actual"] = st.session_state["y_test"].iloc[idx]

    if "packet" in st.session_state:
        packet = st.session_state["packet"]

        col1, col2 = st.columns(2)

        with col1:
            st.subheader("Packet Features")
            st.dataframe(packet)

        with col2:
            pred = st.session_state["best_model"].predict([packet])[0]

            st.subheader("Detection Result")

            if pred == "BENIGN":
                st.success("SAFE (BENIGN)")
            else:
                st.error(f"🚨 ATTACK DETECTED ({pred})")

            st.caption(
                f"Ground Truth Label: {st.session_state['actual']}"
            )

            # ---------------- GROQ AI ----------------
            st.markdown("---")
            st.subheader("Ask AI Analyst")

            if st.button("Explain with AI"):
                if not groq_api_key:
                    st.warning("Please enter Groq API key")
                else:
                    client = Groq(api_key=groq_api_key)

                    prompt = f"""
                    You are a cybersecurity analyst.

                    Prediction: {pred}
                    Actual Label: {st.session_state['actual']}

                    Packet values:
                    {packet.to_string()}

                    Explain why this traffic is benign or an attack.
                    Suggest remediation and preventive measures.
                    Keep it simple for a student.
                    """

                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=[{"role": "user", "content": prompt}],
                        temperature=0.5
                    )

                    st.info(response.choices[0].message.content)

    # ---------------- SHAP EXPLAINABILITY ----------------
    st.header("4️⃣ Explainable AI (SHAP Feature Impact)")

    if st.session_state["best_model_name"] in ["Random Forest", "Decision Tree"]:

        with st.spinner("Generating SHAP explanation..."):

            explainer = shap.TreeExplainer(
                st.session_state["best_model"]
            )

            # Ensure robust sampling of X_test for SHAP explanation
            num_shap_samples = min(200, len(st.session_state["X_test"]))
            if num_shap_samples > 0:
                sample_X = st.session_state["X_test"].sample(n=num_shap_samples, random_state=42)
                shap_values = explainer.shap_values(sample_X)

                fig_shap, ax = plt.subplots(figsize=(10, 5))
                shap.summary_plot(
                    shap_values,
                    sample_X,
                    plot_type="bar",
                    show=False
                )
                st.pyplot(fig_shap)
            else:
                st.info("Not enough samples in X_test to generate SHAP explanation.")

    else:
        st.info(
            "SHAP explanation is supported only for tree-based models."
        )

else:
    st.info("👈 Click **Train & Compare Models** from the sidebar")

2026-02-08 18:06:22.632 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:22.633 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:22.635 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:22.637 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:22.638 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:22.639 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:22.641 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:22.642 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [6]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from groq import Groq

# --------------------------------------------------
# PAGE CONFIG
# --------------------------------------------------
st.set_page_config(
    page_title="AI-Based NIDS | ML Comparison",
    layout="wide"
)

st.title("AI-Based Network Intrusion Detection System")
st.caption("Streamlit App with ML Model Comparison, Attack Detection & Explainable AI")

# --------------------------------------------------
# CONFIG
# --------------------------------------------------
DATA_FILE = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"

FEATURES = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Total Length of Fwd Packets',
    'Fwd Packet Length Max',
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow Packets/s'
]

# --------------------------------------------------
# SIDEBAR
# --------------------------------------------------
st.sidebar.header("🔑 Settings")
groq_api_key = st.sidebar.text_input("Groq API Key", type="password")

# --------------------------------------------------
# LOAD DATA
# --------------------------------------------------
@st.cache_data
def load_data():
    df = pd.read_csv(DATA_FILE, nrows=20000)
    df.columns = df.columns.str.strip()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    return df

df = load_data()
st.sidebar.success(f"Dataset Loaded: {len(df)} rows")

# --------------------------------------------------
# TRAIN BUTTON
# --------------------------------------------------
if st.sidebar.button("🚀 Train & Compare Models"):

    with st.spinner("Training models..."):

        X = df[FEATURES]
        y = df["Label"]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=0.3,
            random_state=42,
            stratify=y
        )

        models = {
            "Logistic Regression": Pipeline([
                ("scaler", StandardScaler()),
                ("model", LogisticRegression(max_iter=1000))
            ]),
            "Decision Tree": DecisionTreeClassifier(max_depth=10),
            "Random Forest": RandomForestClassifier(
                n_estimators=100, max_depth=12, random_state=42
            ),
            "SVM": Pipeline([
                ("scaler", StandardScaler()),
                ("model", SVC(probability=True))
            ]),
            "KNN": Pipeline([
                ("scaler", StandardScaler()),
                ("model", KNeighborsClassifier(n_neighbors=5))
            ])
        }

        results = []

        for name, model in models.items():
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

            report = classification_report(
                y_test, preds, output_dict=True
            )

            results.append({
                "Model": name,
                "Accuracy": accuracy_score(y_test, preds),
                "Precision": report["weighted avg"]["precision"],
                "Recall": report["weighted avg"]["recall"],
                "F1-Score": report["weighted avg"]["f1-score"]
            })

        results_df = pd.DataFrame(results).sort_values(
            "Accuracy", ascending=False
        )

        best_model_name = results_df.iloc[0]["Model"]
        best_model = models[best_model_name]
        best_model.fit(X_train, y_train)

        st.session_state.update({
            "results_df": results_df,
            "best_model": best_model,
            "best_model_name": best_model_name,
            "X_test": X_test,
            "y_test": y_test
        })

# --------------------------------------------------
# DISPLAY RESULTS
# --------------------------------------------------
if "results_df" in st.session_state:

    # ---------------- MODEL COMPARISON TABLE ----------------
    st.header("1️⃣ Machine Learning Model Comparison")
    st.dataframe(
        st.session_state["results_df"],
        use_container_width=True
    )

    st.success(
        f"🏆 Best Model Selected: "
        f"**{st.session_state['best_model_name']}**"
    )

    # ---------------- BAR CHART ----------------
    st.subheader("📊 Model Performance Comparison")

    metrics = ["Accuracy", "Precision", "Recall", "F1-Score"]
    melted = st.session_state["results_df"].melt(
        id_vars="Model",
        value_vars=metrics,
        var_name="Metric",
        value_name="Score"
    )

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=melted,
        x="Metric",
        y="Score",
        hue="Model",
        ax=ax
    )
    ax.set_ylim(0, 1)
    ax.set_title("ML Model Performance Comparison")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    st.pyplot(fig)

    # ---------------- CONFUSION MATRIX ----------------
    st.subheader("Confusion Matrix (Best Model)")
    y_pred = st.session_state["best_model"].predict(
        st.session_state["X_test"]
    )

    cm = confusion_matrix(
        st.session_state["y_test"], y_pred
    )

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=np.unique(st.session_state["y_test"]),
        yticklabels=np.unique(st.session_state["y_test"])
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    st.pyplot(fig)

    # ---------------- ATTACK VS BENIGN ----------------
    st.header("2️⃣ Attack vs Benign Examples")

    benign = df[df["Label"] == "BENIGN"].sample(1)
    attack = df[df["Label"] != "BENIGN"].sample(1)

    col1, col2 = st.columns(2)

    with col1:
        st.subheader("🟢 BENIGN Packet")
        st.dataframe(benign[FEATURES].T)
        st.success("Normal traffic behavior")

    with col2:
        st.subheader("🔴 ATTACK Packet")
        st.dataframe(attack[FEATURES].T)
        st.error("Malicious traffic behavior")

    # ---------------- LIVE SIMULATION ----------------
    st.header("3️⃣ Live Packet Detection")

    if st.button("🎲 Capture Random Packet"):
        idx = np.random.randint(0, len(st.session_state["X_test"]))
        st.session_state["packet"] = st.session_state["X_test"].iloc[idx]
        st.session_state["actual"] = st.session_state["y_test"].iloc[idx]

    if "packet" in st.session_state:
        packet = st.session_state["packet"]

        col1, col2 = st.columns(2)

        with col1:
            st.subheader("Packet Features")
            st.dataframe(packet)

        with col2:
            pred = st.session_state["best_model"].predict([packet])[0]

            st.subheader("Detection Result")

            if pred == "BENIGN":
                st.success("SAFE (BENIGN)")
            else:
                st.error(f"🚨 ATTACK DETECTED ({pred})")

            st.caption(
                f"Ground Truth Label: {st.session_state['actual']}"
            )

            # ---------------- GROQ AI ----------------
            st.markdown("---")
            st.subheader("Ask AI Analyst")

            if st.button("Explain with AI"):
                if not groq_api_key:
                    st.warning("Please enter Groq API key")
                else:
                    client = Groq(api_api_key=groq_api_key)

                    prompt = f"""
                    You are a cybersecurity analyst.

                    Prediction: {pred}
                    Actual Label: {st.session_state['actual']}

                    Packet values:
                    {packet.to_string()}

                    Explain why this traffic is benign or an attack.
                    Suggest remediation and preventive measures.
                    Keep it simple for a student.
                    """

                    response = client.chat.completions.create(
                        model="llama-3.3-70b-versatile",
                        messages=[{"role": "user", "content": prompt}],
                        temperature=0.5
                    )

                    st.info(response.choices[0].message.content)

    # ---------------- SHAP EXPLAINABILITY ----------------
    st.header("4️⃣ Explainable AI (SHAP Feature Impact)")

    best_model = st.session_state["best_model"]
    best_model_name = st.session_state["best_model_name"]

    # SHAP only for tree-based models
    if best_model_name in ["Random Forest", "Decision Tree"]:

        with st.spinner("Generating SHAP explanation..."):

            # Handle Pipeline vs direct model
            if hasattr(best_model, "named_steps"):
                tree_model = best_model.named_steps.get("model", None)
            else:
                tree_model = best_model

            # Ensure robust sampling of X_test for SHAP explanation
            num_shap_samples = min(200, len(st.session_state["X_test"]))
            if num_shap_samples > 0:
                X_sample = st.session_state["X_test"].sample(n=num_shap_samples, random_state=42)

                # Create SHAP explainer
                explainer = shap.TreeExplainer(tree_model)

                # Compute SHAP values
                shap_values = explainer.shap_values(X_sample)

                # ----- MULTI-CLASS HANDLING ----- TODO: Make this dynamic
                if isinstance(shap_values, list):
                    # Pick BENIGN class if exists, else first class
                    class_names = tree_model.classes_
                    if "BENIGN" in class_names:
                        class_index = list(class_names).index("BENIGN")
                    else:
                        class_index = 0

                    shap_values_to_plot = shap_values[class_index]
                else:
                    shap_values_to_plot = shap_values

                # ----- FORCE MATPLOTLIB PLOT -----
                fig, ax = plt.subplots(figsize=(10, 5))

                shap.summary_plot(
                    shap_values_to_plot,
                    X_sample,
                    plot_type="bar",
                    show=False
                )

                st.pyplot(fig)
            else:
                st.info("Not enough samples in X_test to generate SHAP explanation.")

    else:
        st.info(
            "SHAP visualization is available only for "
            "Random Forest or Decision Tree models."
        )

else:
    st.info("👈 Click **Train & Compare Models** from the sidebar")

2026-02-08 18:06:32.398 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:32.402 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:32.403 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:32.405 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:32.408 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:32.412 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:32.413 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:32.416 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [7]:
import streamlit as st
import pandas as pd
import numpy as np

# Removed st.cache_data.clear() as it might not be needed in standalone script,
# and @st.cache_data decorator removed for non-Streamlit context.
def load_data(): # Renamed for clarity with UNSW-NB15 dataset
    df = pd.read_csv(
        "/content/UNSW_NB15_training-set.csv", # Updated to UNSW-NB15 dataset
        encoding='ISO-8859-1', # Often needed for this dataset
        low_memory=False
    )

    # Clean column names
    df.columns = df.columns.str.strip().str.lower()

    # Before replacing inf/nan, check original labels
    print(f"Unique labels before processing: {df['attack_cat'].unique()}")

    # Replace infinite values with NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Fill NaN values with 0
    df.fillna(0, inplace=True)

    # Convert the 'attack_cat' column to string type to handle mixed types
    df['attack_cat'] = df['attack_cat'].astype(str)

    print(f"Unique labels after processing: {df['attack_cat'].unique()}")
    print(f"Value counts after processing:\n{df['attack_cat'].value_counts()}")

    return df

In [8]:
# Download the dataset if not already present
import os

data_file_path = "/content/UNSW_NB15_training-set.csv"
if not os.path.exists(data_file_path):
    print(f"Downloading {os.path.basename(data_file_path)}...")
    !wget -q -O {data_file_path} https://raw.githubusercontent.com/easy-deep/datasets/main/UNSW_NB15_training-set.csv
    print("Download complete.")
else:
    print(f"{os.path.basename(data_file_path)} already exists.")

UNSW_NB15_training-set.csv already exists.


In [9]:
df = load_data() # Call the renamed function
# Removed st.sidebar.success as it causes issues outside Streamlit app context
print(f"Dataset Loaded: {len(df)} rows")

Unique labels before processing: ['Normal' 'Reconnaissance' 'Backdoor' 'DoS' 'Exploits' 'Analysis'
 'Fuzzers' 'Worms' 'Shellcode' 'Generic' nan]
Unique labels after processing: ['Normal' 'Reconnaissance' 'Backdoor' 'DoS' 'Exploits' 'Analysis'
 'Fuzzers' 'Worms' 'Shellcode' 'Generic' '0']
Value counts after processing:
attack_cat
Normal            33591
Generic           18871
Exploits          11132
Fuzzers            6062
DoS                4089
Reconnaissance     3496
Analysis            677
Backdoor            583
Shellcode           378
Worms                44
0                     1
Name: count, dtype: int64
Dataset Loaded: 78924 rows


In [10]:
FEATURES = [
    'dur',        # Duration
    'spkts',      # Source packets
    'dpkts',      # Destination packets
    'sbytes',     # Source bytes
    'dbytes',     # Destination bytes
    'sttl',       # Source TTL
    'dttl',       # Destination TTL
    'sload',      # Source load
    'dload'       # Destination load
]

# Filter out rows where 'Label' is '0' as it's a single-sample class causing issues with stratification
df_filtered = df[df['attack_cat'] != '0'].copy() # Use 'attack_cat' for UNSW-NB15 labels

X = df_filtered[FEATURES]
y = df_filtered['attack_cat']

In [11]:
from sklearn.model_selection import train_test_split

# Filter out any single-sample classes that might appear after filtering '0'
# This ensures that all classes in y have at least 2 samples for stratification
value_counts = y.value_counts()
single_sample_classes = value_counts[value_counts < 2].index.tolist()
if single_sample_classes:
    print(f"Removing single-sample classes for stratification: {single_sample_classes}")
    y_filtered_for_split = y[~y.isin(single_sample_classes)]
    X_filtered_for_split = X.loc[y_filtered_for_split.index]
else:
    y_filtered_for_split = y
    X_filtered_for_split = X

# Proceed with train_test_split if there are still multiple classes after filtering
if len(y_filtered_for_split.unique()) > 1:
    X_train, X_test, y_train, y_test = train_test_split(
        X_filtered_for_split,
        y_filtered_for_split,
        test_size=0.3,
        random_state=42,
        stratify=y_filtered_for_split
    )
    print("Data split successfully with stratification.")
elif len(y_filtered_for_split.unique()) == 1 and len(y_filtered_for_split) > 1:
    print("Warning: Only one class remaining after filtering, splitting without stratification.")
    X_train, X_test, y_train, y_test = train_test_split(
        X_filtered_for_split,
        y_filtered_for_split,
        test_size=0.3,
        random_state=42
    )
else:
    print("Error: Not enough data or classes left to perform a meaningful split.")
    X_train, X_test, y_train, y_test = None, None, None, None # Ensure these are set to None if split fails

Data split successfully with stratification.


In [12]:
st.write("Sample UNSW_NB15_training set Rows")
st.dataframe(df.head())

st.write("Feature Statistics")
st.dataframe(X.describe())


2026-02-08 18:06:59.402 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:59.404 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:59.405 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:59.417 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:59.418 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:59.419 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:59.420 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-08 18:06:59.422 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [13]:
!pip install pyngrok

In [14]:
from pyngrok import ngrok

# Replace with your ngrok authtoken
# You can get one from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "39ErPZzAT5coOll2PmZpA9QrWPE_Dh5Q9cmudR9KcsZCMwq1" # <--- PASTE YOUR TOKEN HERE

if NGROK_AUTH_TOKEN == "YOUR_ACTUAL_NGROK_AUTH_TOKEN_HERE":
    print("WARNING: Please replace 'YOUR_ACTUAL_NGROK_AUTH_TOKEN_HERE' with your actual ngrok authtoken.")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authtoken set.")

ngrok authtoken set.


In [ ]:
from pyngrok import ngrok
import subprocess
import os
import time

# Replace with your ngrok authtoken
# You can get one from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "39ErPZzAT5coOll2PmZpA9QrWPE_Dh5Q9cmudR9KcsZCMwq1" # <--- PASTE YOUR TOKEN HERE

if NGROK_AUTH_TOKEN == "YOUR_ACTUAL_NGROK_AUTH_TOKEN_HERE":
    print("WARNING: Please replace 'YOUR_ACTUAL_NGROK_AUTH_TOKEN_HERE' with your actual ngrok authtoken.")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authtoken set.")


# Ensure all ngrok processes are killed to free up resources and tunnels
print("Attempting to kill all ngrok processes...")
subprocess.run(["killall", "ngrok"], capture_output=True, text=True)
ngrok.kill() # pyngrok's internal kill mechanism
time.sleep(2) # Give ngrok processes a moment to fully terminate
print("ngrok processes killed (if any were running). ")

# Disconnect any previously active tunnels that might still be reported by pyngrok
# This helps in cases where ngrok.kill() might not immediately clear all server-side tunnel registrations.
try:
    for tunnel in ngrok.get_tunnels():
        print(f"Found and disconnecting existing tunnel: {tunnel.public_url}")
        ngrok.disconnect(tunnel.public_url)
except Exception as e:
    # This might happen if no tunnels are active, which is fine.
    print(f"No active tunnels found or error during initial disconnection attempt: {e}")
time.sleep(1) # Small pause after disconnecting tunnels

# Start Streamlit in the background
streamlit_process = subprocess.Popen(["python", "-m", "streamlit", "run", "app.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# Give Streamlit a moment to start up
time.sleep(5)

# Get the local port Streamlit is running on (usually 8501)
streamlit_port = 8501

# Open a ngrok tunnel to the Streamlit port
public_url = None # Initialize to None in case connect fails
try:
    public_url = ngrok.connect(streamlit_port)
    print(f"Streamlit App URL: {public_url}")
except Exception as e:
    print(f"Failed to establish ngrok tunnel: {e}")
    print("It's possible a tunnel is still active or there's an issue with ngrok itself.")
    print("Please try stopping the current cell execution, ensure no other ngrok or Streamlit processes are running, and then re-run this cell.")
    # If connection fails, ensure Streamlit process is terminated to prevent lingering apps
    if streamlit_process.poll() is None: # Check if Streamlit process is still running
        streamlit_process.terminate()
        streamlit_process.wait()
    raise # Re-raise the exception to stop execution and signal error

# Keep the cell alive and Streamlit running
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopping Streamlit and ngrok tunnel...")
    if public_url:
        ngrok.disconnect(public_url)
    streamlit_process.terminate()
    streamlit_process.wait()
    print("Streamlit and ngrok stopped.")

ngrok authtoken set.
Attempting to kill all ngrok processes...
ngrok processes killed (if any were running). 
Streamlit App URL: NgrokTunnel: "https://unparasitical-eda-transovarian.ngrok-free.dev" -> "http://localhost:8501"


The following code block demonstrates how SHAP values are calculated and plotted for explainable AI. This code is extracted from the existing Streamlit application logic.